> ### Public copy (demo data)
>
> The client's 2024 survey cannot be shared publicly, so it is **not in this repository**. Instead,
> `scripts/make_demo_workbook.py` creates `data/raw/engagement_survey_2024.xlsx`, a stand-in file
> with the same structure as the real one: the same sheet name, 22 columns, 42 rows, category
> headers, two 1-10 items and the same delta-from-company-mean format. Only the numbers are
> **invented**.
>
> This means the code, the method and the reasoning below are the originals, but every number,
> table and chart comes from demo data. Phase 1 figures are labelled **DEMO** instead of REAL.
> Where a result depends on the client's numbers, the text says what to look for instead of
> quoting them. Everything the notebooks produce is written to `outputs/`.
>
> To run on the real data, put the client workbook in `data/raw/` and set `data.provenance` to
> `real` in `config/config.yaml`. The labels then switch back.


# NB01 — Data Loading & Quality Treatment

| | |
|---|---|
| **Purpose** | Load the 2024 workbook, verify it is understood, and produce the analysis-ready dataset. |
| **Objectives** | Groundwork for all four. The refined objectives RO1–RO4 are defined in the README. |
| **Input** | `data/raw/engagement_survey_2024.xlsx` |
| **Outputs** | `data/processed/dept_item_matrix.csv` · `tab01` · `tab02` |

**What this file is and is not.** An *aggregated* summary, not raw responses:
`Company Overall (107)` holds item means across 107 respondents; every department column holds
a **delta** from that mean; 8 of the 42 rows are category headers; the `manager` column is
empty. This shape bounds everything Phase 1 may claim.

In [1]:
# Environment — package installed via `pip install -e .`; path fallback for direct runs
import sys
sys.path.insert(0, "../src")

import pandas as pd
from engagement import io, quality
from engagement.config import load_config, get_seed

cfg = load_config()
pd.set_option("display.max_colwidth", 90)
print(f"Config loaded · global seed = {get_seed(cfg)}")

Config loaded · global seed = 5496


---
## 1 · Load and verify structure

**Decision box**

| | |
|---|---|
| **What** | Read the workbook and check its documented structure programmatically (label column, company-mean column, all 8 category headers present). |
| **Why this method** | Fail-loudly approach halts the pipeline immediately upon detecting a data error, which saves the client time and resources. |
| **Alternative rejected** | Executing read_excel directly and manually, without any validation. |
| **Feeds** | The tidy dataframe used by every later step. |

In [2]:
raw = io.load_heatmap(cfg)          # raises with a clear message if structure differs
raw.head(8)

,Category / Question,Company Overall (107),Other (24),A,B,C,D,E,F,G,...,J,K,L,M,N,O,P,Q,R,manager
0,Working for AccessFintech,4.59,-0.39,-0.26,0.01,-0.23,-0.26,0.19,-0.95,-0.43,...,-1.21,-0.79,-1.14,-0.32,0.11,-0.65,0.02,-0.14,-0.06,NaN
1,I can see myself working for Access Fintech for the next 2 years.,3.99,-0.53,-0.08,0.03,-0.66,-0.23,0.58,-0.62,-0.47,...,-0.83,-0.90,-1.01,-0.53,-0.09,-0.58,-0.17,0.19,0.07,NaN
2,I am optimistic about Access Fintech's long-term success.,4.11,-0.35,-0.20,-0.11,-0.07,0.00,0.31,-0.50,-0.46,...,-0.81,-0.70,-0.78,-0.53,0.35,-0.54,0.32,0.06,-0.22,NaN
3,"As an employee, I get sufficient and relevant information about company updates/org ch...",4.07,-0.32,-0.76,0.04,-0.51,-0.23,0.02,-0.71,-0.42,...,-0.76,-0.93,-0.76,-0.15,-0.24,0.00,0.03,-0.26,0.16,NaN
4,The day-to-day decisions here demonstrate that we continuously learn from experiences ...,4.26,-0.59,-0.11,-0.03,0.31,0.06,0.06,-0.68,-0.75,...,-0.58,-0.51,-0.68,-0.36,0.17,-0.81,0.46,-0.10,-0.10,NaN
5,I would recommend AFT to my friends.,3.71,-0.46,-0.24,-0.30,-0.38,-0.39,-0.09,-0.74,-0.68,...,-0.91,-0.35,-1.06,-0.26,0.09,-0.20,-0.01,-0.27,-0.16,NaN
6,How happy are you at AccessFintech on a scale of 1 to 10?,7.40,-0.06,-0.17,0.40,-0.04,-0.78,0.24,-2.44,0.18,...,-3.36,-1.38,-2.52,-0.08,0.38,-1.75,-0.49,-0.47,-0.11,NaN
7,My role at Access Fintech,3.82,-0.29,-0.10,-0.20,-0.61,-0.14,-0.21,-0.29,-0.24,...,-0.93,-0.71,-0.31,-0.08,0.01,-0.47,-0.31,0.21,-0.68,NaN


**Decision box**

| | |
|---|---|
| **What** | Reshape the wide heatmap into tidy long format: one row per (department × item), tagged with category, scale and value type. |
| **Why this method** | One observation per row makes joins, group-bys, plotting and testing straightforward, and turns each transformation into a testable function. |
| **Alternative rejected** | Analysing the Excel-shaped table in place: fragile positional indexing, and untestable. |
| **Feeds** | All Phase 1 functions. |

In [3]:
tidy = io.tidy(raw, cfg)
print(f"{len(tidy)} records · {tidy[~tidy['is_category_header']]['item'].nunique()} items · "
      f"{tidy['department'].nunique()-1} department units (+ Company)")
tidy.head()

840 records · 34 items · 19 department units (+ Company)


,category,item,is_category_header,scale,department,value,value_type
0,Working for AccessFintech,Working for AccessFintech,True,5,Company,4.59,mean
1,Working for AccessFintech,Working for AccessFintech,True,5,Other (24),-0.39,delta
2,Working for AccessFintech,Working for AccessFintech,True,5,A,-0.26,delta
3,Working for AccessFintech,Working for AccessFintech,True,5,B,0.01,delta
4,Working for AccessFintech,Working for AccessFintech,True,5,C,-0.23,delta


---
## 2 · Inspection branch: "is the workbook internally sound?"

The pipeline splits here: inspecting the data and repairing it are
separate jobs with separate evidence. Inspection runs **on the raw scale, before any
treatment**, so that any defect remains visible to check rather than being silently corrected away.

**Decision box**

| | |
|---|---|
| **What** | Check each theme total against the average of its own questions, and flag any theme that mixes the 1–5 and 1–10 scales. |
| **Why this method** | Two separate questions: does the total add up, and can it be compared with the other themes? A theme total can pass the first and fail the second. |
| **Alternative rejected** | Trusting the workbook's own category rows. The check below shows why that is unsafe. |
| **Feeds** | `tab01`; the decision not to reuse any workbook roll-up. |

In [4]:
structure_report = quality.validate_structure(tidy, cfg)
structure_report

,category,header_value,computed_item_mean,difference,within_tolerance,note
0,Working for AccessFintech,4.59,4.590,0.000,True,mixes 1-5 and 1-10 items
1,My role at Access Fintech,3.82,3.823,-0.003,True,
2,Culture & Wellbeing,3.81,3.810,0.000,True,
3,Work/Life Balance,4.85,4.853,-0.003,True,mixes 1-5 and 1-10 items
4,Working with my Line Manager,3.88,3.884,-0.004,True,
5,Working with my Team,4.20,4.200,0.000,True,
6,Working with the Leadership Teams (Heads of Department),3.27,3.263,0.007,True,
7,Working with the Executive Teams,3.07,3.073,-0.003,True,


**Finding.** All eight headers match the mean of their own items, within the configured 0.05 tolerance, so the roll-ups are arithmetically sound.

But two categories: *Working for AccessFintech* and *Work/Life Balance* are mixed 1–5 items with a 1–10 item, which inflates their means. Arithmetically correct is not the same as usable.

**Consequence: no workbook roll-up is reused; every category score downstream is recomputed from harmonised items.**

---
## 3 · Treatment branch: producing the analysis-ready dataset

**Decision box**

| | |
|---|---|
| **What** | Recover absolute scores: `absolute = company mean + department delta`. |
| **Why this method** | Deltas are uninterpretable in isolation; absolute scores give a 19 × 34 matrix that any reader can benchmark directly. |
| **Alternative rejected** | Analysing deltas as-is, which loses the level information needed for cross-item comparison. |
| **Feeds** | The department × item matrix behind every Phase 1 analysis. |

In [5]:
absolute = quality.reconstruct_absolute(tidy)
# Company rows must survive untouched — asserted here, also covered in tests/test_quality.py
assert (absolute[absolute.department == "Company"].abs_value
        == absolute[absolute.department == "Company"].value).all()
absolute.head(3)

,category,item,is_category_header,scale,department,value,value_type,company_mean,abs_value
0,Working for AccessFintech,I can see myself working for Access Fintech for the next 2 years.,False,5,Company,3.99,mean,3.99,3.99
1,Working for AccessFintech,I can see myself working for Access Fintech for the next 2 years.,False,5,Other (24),-0.53,delta,3.99,3.46
2,Working for AccessFintech,I can see myself working for Access Fintech for the next 2 years.,False,5,A,-0.08,delta,3.99,3.91


**Decision box**

| | |
|---|---|
| **What** | Rescale the two 1–10 items onto the 1–5 metric: **v₅ = 1 + (v₁₀ − 1) × 4/9**. |
| **Why this method** | The unique linear map preserving both endpoints (min–max rescaling; cf. Cohen et al., 1999). Items on different scales cannot validly be averaged, as the workbook itself demonstrates. |
| **Alternative rejected** | Z-standardisation (implemented and config-switchable) is statistically cleaner, but it changes the 1–5 scale the client reads. |
| **Feeds** | Every score used from this point is the `harmonised` column. |

In [6]:
harmonised = quality.harmonise_scales(absolute, cfg)
harmonised[harmonised["scale"] == 10][["item", "department", "abs_value", "harmonised"]].head(4)

,item,department,abs_value,harmonised
100,How happy are you at AccessFintech on a scale of 1 to 10?,Company,7.40,3.844444
101,How happy are you at AccessFintech on a scale of 1 to 10?,Other (24),7.34,3.817778
102,How happy are you at AccessFintech on a scale of 1 to 10?,A,7.23,3.768889
103,How happy are you at AccessFintech on a scale of 1 to 10?,B,7.80,4.022222


**Residual risk.** The linear map assumes interval-scale equivalence, with a contested
assumption for Likert data, recorded in `tab02` rather than hidden.

---
## 4 · Consolidated quality evidence

Issue → how it was detected → how it was treated → what risk remains. This single table is the
evidence that analytical decisions reflect the data's integrity problems; it is saved as `tab02`.

In [7]:
summary = quality.quality_summary(structure_report)
summary

,issue,detection,treatment,residual_risk
0,"Departments reported as deltas, not scores",Workbook inspection (io.tidy),Reconstruct absolute = company mean + delta,Rounding in source deltas (~±0.01)
1,Mixed 1-5 and 1-10 response scales,Scale audit against item wording,Linear rescale of 1-10 items onto 1-5 (config-switchable to z-score),Linear map assumes interval-scale equivalence
2,Category roll-ups average across mixed scales,validate_structure: 2 category roll-up(s) average across mixed scales (headers arithme...,All category scores recomputed from harmonised items; workbook headers not trusted,"None — headers replaced, not repaired"
3,Empty 'manager' column,Missing-value map (io.tidy),Documented as data-supply gap; excluded from analysis,Manager-level analysis impossible until 2026 redesign
4,"Aggregated single-wave data, no demographics",Data appraisal (Section 1 of report),Phase 1 bounded to description; Phase 2 synthetic demonstration,Drivers/risk/trends unanswerable on real data — by design


In [8]:
from pathlib import Path
t = Path(cfg["paths"]["tables_dir"]); t.mkdir(parents=True, exist_ok=True)
io.export_processed(quality.to_matrix(harmonised, "harmonised"), cfg, "dept_item_matrix")
structure_report.to_csv(t / "tab01_structure_validation.csv", index=False)
summary.to_csv(t / "tab02_quality_summary.csv", index=False)
print("data/processed/dept_item_matrix.csv · tab01 · tab02")

data/processed/dept_item_matrix.csv · tab01 · tab02


---
## Boundary statement

This aggregated, single-wave, demographically thin file supports **description only**. It
cannot establish which factors drive engagement, quantify individual risk, show change over
time, or segment by demographics. That boundary is the design brief for Phase 2.

**Next:** `02_phase1_diagnostic.ipynb`

---
## References

Cohen, P., Cohen, J., Aiken, L.S. and West, S.G. (1999) 'The problem of units and the
circumstance for POMP', *Multivariate Behavioral Research*, 34(3), pp. 315–346.
doi:10.1207/S15327906MBR3403_2.